# CO₂ surrogate training and solver check

Run this notebook in Kaggle with **GPU enabled**. Add the Kaggle notebook output containing `co2_xz_dataset_v2` as an input. This notebook trains CCS and CCS-GNN, plans on one held-out validation geology, then checks the planned schedule with the CO₂ solver.

The game scenario is exploratory with normalized utility units. It is not a field operating plan or a pressure safety limit. The solver check uses CPU; GPU is used for training and surrogate planning.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/kaggle/working/shale-well-game-theory-codes')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--single-branch',
        '--branch', 'codex/ccs-lstm-fno-variants',
        'https://github.com/smerarawal/shale-well-game-theory-codes.git',
        str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

candidates = [
    p.parent for p in Path('/kaggle/input').rglob('dataset_config.json')
    if (p.parent / 'train' / 'friendly.npz').is_file()
]
print('CO₂ dataset candidates:')
for path in candidates:
    print(path)
assert candidates, 'Attach the CO₂ transition dataset with train/validation/test folders.'
assert len(candidates) == 1, 'Several candidates found; set DATA to the intended co2_xz_dataset_v2 folder.'
DATA = candidates[0]
SCENARIO = REPO / 'sequential_game' / 'co2_surrogate_scenario_exploratory.json'
OUT = Path('/kaggle/working/co2_surrogate_refined')
OUT.mkdir(parents=True, exist_ok=True)
print('Using dataset:', DATA)
print('Scenario:', SCENARIO)
print('Output:', OUT)

In [ ]:
import torch

assert torch.cuda.is_available(), 'In Kaggle, select Settings → Accelerator → GPU, then restart and run this notebook.'
with open(DATA / 'dataset_config.json', encoding='utf-8') as f:
    dataset_config = json.load(f)
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset transitions:', dataset_config['transitions'])
print('Dataset stages:', dataset_config['parameters']['stages'])
print('Dataset stage days:', dataset_config['parameters']['stage_days'])
for split in ('train', 'validation', 'test'):
    assert (DATA / split / 'friendly.npz').is_file(), f'Missing {split}/friendly.npz'
print('Dataset split files found.')

## Train the matched surrogate candidates

This runs two backbones across three seeds using six-stage pushforward training and pressure-tail-aware validation. It saves checkpoints and metrics under `/kaggle/working/co2_surrogate_refined`. Do not interrupt this cell if you want all six model-seed runs completed.

In [ ]:
subprocess.run([
    sys.executable,
    str(REPO / 'sequential_game' / 'train_co2_xz_surrogate.py'),
    '--data-dir', str(DATA),
    '--out-dir', str(OUT),
    '--backbones', 'ccs', 'ccs_gnn',
    '--seeds', '0', '1', '2',
    '--epochs', '80',
    '--patience', '12',
    '--rollout-unroll', '6',
    '--rollout-validation-weight', '0.25',
    '--rollout-validation-every', '5',
    '--valid-loss-weight', '0.1',
    '--mass-loss-weight', '0.1',
    '--pmax-loss-weight', '0.5',
    '--pressure-tail-loss-weight', '0.25',
    '--pressure-tail-fraction', '0.1',
    '--device', 'cuda',
], check=True)

checkpoints = sorted(OUT.glob('*.pt'))
print('Saved checkpoints:', [p.name for p in checkpoints])
assert (OUT / 'co2_ccs_s0.pt').is_file(), 'Expected CCS seed-0 checkpoint was not created.'

## Plan on a held-out validation geology

This uses CCS seed 0 and validation geology seed 41007. The planner receives the scenario values as command-line arguments; it does not load the JSON file directly. The pressure threshold and penalty are exploratory model parameters, not a geomechanical safety limit.

In [ ]:
PLAN = Path('/kaggle/working/co2_surrogate_plan.json')
subprocess.run([
    sys.executable,
    str(REPO / 'sequential_game' / 'co2_differentiable_game.py'),
    '--data-dir', str(DATA),
    '--checkpoint', str(OUT / 'co2_ccs_s0.pt'),
    '--cohort', 'friendly',
    '--split', 'validation',
    '--geology-seed', '41007',
    '--out', str(PLAN),
    '--device', 'cuda',
    '--horizon', '6',
    '--stage-days', '30',
    '--discount-factor', '1.0',
    '--rate-max-m3-s', '1e-7',
    '--pressure-cap-pa', '5000000',
    '--pressure-damage-per-pa2', '1e-13',
    '--storage-value-per-kg', '1.0',
    '--pipeline-capacity-kg-per-stage', '252.72',
    '--pipeline-dispatch-rule', 'proportional',
], check=True)

plan = json.loads(PLAN.read_text(encoding='utf-8'))
print('Planned rate schedule [stage][operator], m³/s:')
print(json.dumps(plan['rates_m3_s_by_stage_operator'], indent=2))

## Re-evaluate the schedule with the solver

The solver replay is the physical reference for this schedule. It runs on CPU. The validation NPZ is ordered with seed 41007 first for the friendly validation cohort, so row 0 is used here.

In [ ]:
SOLVER_CHECK = Path('/kaggle/working/co2_solver_check.json')
subprocess.run([
    sys.executable,
    str(REPO / 'sequential_game' / 'co2_solver_game_search.py'),
    'evaluate',
    '--geology', str(DATA / 'validation' / 'friendly.npz'),
    '--row', '0',
    '--config', str(SCENARIO),
    '--schedule', str(PLAN),
    '--out', str(SOLVER_CHECK),
], check=True)

solver_result = json.loads(SOLVER_CHECK.read_text(encoding='utf-8'))
print('Solver project resource welfare:', solver_result['project_resource_welfare'])
print('Solver operator payoffs:', solver_result['operator_payoff'])
print('Solver runtime (seconds):', solver_result['runtime_seconds'])
print('Maximum stage pressure (Pa):', max(r['pressure_max_pa'] for r in solver_result['stage_records']))
print('Maximum absolute mass residual (kg):', max(abs(r['mass_balance_residual_kg']) for r in solver_result['stage_records']))
print('Full solver report:', SOLVER_CHECK)

## Save outputs

Keep the Kaggle notebook version with its output. Important files are the six model checkpoints and training metrics in `/kaggle/working/co2_surrogate_refined`, `co2_surrogate_plan.json`, and `co2_solver_check.json`. Check the pressure-error and pressure-violation metrics carefully: the attached dataset has a high pmax tail, and its mass-balance closure does not by itself validate pressure accuracy.